# BudgetBot — governed agent tool payments

LangChain-style agent loop with **BudgetGuard** + **JustificationGuard**.

**Run:** Execute cells top-to-bottom. Uses mock chain settlement — no TestNet funds needed.

**Prereq:** `pip install -e "./python[dev]" jupyter` from repo root.

In [ ]:
import sys
from pathlib import Path

# Notebook lives in python/examples/community/budgetbot/
sys.path.insert(0, str(Path("../..").resolve()))
from _notebook_utils import FAKE_RECIPIENT, enable_mock_router, display_ledger

from algopay import AlgoPay
from algopay.core.types import Network

DAILY_LIMIT = "0.10"
TOOLS = [
    {"name": "search_web", "cost": "0.02", "purpose": "Search flights to Tokyo"},
    {"name": "weather_api", "cost": "0.01", "purpose": "Weather forecast for trip dates"},
    {"name": "hotel_api", "cost": "0.05", "purpose": "Hotel availability check"},
    {"name": "translate", "cost": "0.01", "purpose": ""},
    {"name": "premium_data", "cost": "0.50", "purpose": "Market data feed"},
]

client = AlgoPay(network=Network.ALGORAND_TESTNET)
enable_mock_router(client)
print("Mock mode: no TestNet funds required.")

In [ ]:
ws = client.wallet.create_wallet_set("budgetbot")
wallet = client.wallet.create_wallet(ws.id)

await client.add_budget_guard(wallet.id, daily_limit=DAILY_LIMIT, name="daily_cap")
await client.add_justification_guard(wallet.id, min_length=8, name="why")

print(f"Wallet: {wallet.address[:16]}...")
print(f"Guards: daily {DAILY_LIMIT} USDC, justification min 8 chars")

In [ ]:
results = []
for tool in TOOLS:
    purpose = tool.get("purpose") or None
    r = await client.pay(wallet.id, FAKE_RECIPIENT, tool["cost"], purpose=purpose)
    results.append({
        "tool": tool["name"],
        "amount": tool["cost"],
        "status": "ok" if r.success else r.status.value,
        "detail": r.error or "",
    })

try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(results))
except ImportError:
    for row in results:
        print(row)

In [ ]:
entries = await client.ledger.query(wallet_id=wallet.id, limit=20)
print(f"Ledger ({len(entries)} entries)")
display_ledger(entries)